In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('All libraries imported successfully!')

In [ ]:

# Load the CSV
df = pd.read_csv('Mall_Customers.csv')

# First 5 rows
print('First 5 rows:')
print(df.head())

# Shape
print(f'\nDataset Shape: {df.shape}')

# Check for missing values
print('\nMissing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Select only Annual Income and Spending Score as NumPy array
# These two features are chosen to keep clusters 2D and visually interpretable
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].values

print(f'Feature matrix shape: {X.shape}')  # should be (200, 2)
print(f'\nFirst 5 rows of X:')
print(X[:5])

In [ ]:
# Z-score standardization from scratch using NumPy
# Store mean and std
feature_mean = X.mean(axis=0)
feature_std  = X.std(axis=0)

X_scaled = (X - feature_mean) / feature_std

print(f'Per-feature mean (original): {feature_mean}')
print(f'Per-feature std  (original): {feature_std}')
print(f'\nAfter scaling:')
print(f'  Mean: {X_scaled.mean(axis=0).round(6)}  (should be ~0)')
print(f'  Std : {X_scaled.std(axis=0).round(6)}   (should be ~1)')

In [ ]:
# Baseline scatter plot — raw data before clustering
plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], color='steelblue', alpha=0.6, edgecolors='white', s=60)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.title('Raw Data — Before Clustering (Baseline Visualization)')
plt.tight_layout()
plt.show()

In [ ]:
def initialize_centroids(X, K, seed=42):
    """
    Randomly select K distinct rows from X as initial centroids.

    Args:
        X (np.ndarray): Data matrix of shape (n_samples, n_features).
        K (int): Number of clusters.
        seed (int): Random seed for reproducibility.

    Returns:
        np.ndarray: Initial centroid array of shape (K, n_features).
    """
    np.random.seed(seed)
    # Randomly pick K unique row indices
    indices = np.random.choice(len(X), size=K, replace=False)
    return X[indices].copy()



test_centroids = initialize_centroids(X_scaled, K=5, seed=42)
print(f'Initial centroids shape: {test_centroids.shape}')  # should be (5, 2)
print('Initial centroids (scaled):')
print(test_centroids)

In [ ]:
def assign_clusters(X, centroids):
    """
    Assign each data point to the nearest centroid using Euclidean distance.

    Args:
        X (np.ndarray): Data matrix of shape (n_samples, n_features).
        centroids (np.ndarray): Current centroid array of shape (K, n_features).

    Returns:
        np.ndarray: 1-D integer array of cluster assignments of length n_samples.
    """
    labels = []
    for point in X:
        # Euclidean distance from this point to every centroid
        distances = np.sqrt(np.sum((centroids - point) ** 2, axis=1))
        labels.append(np.argmin(distances))  # assign to nearest centroid
    return np.array(labels)



test_labels = assign_clusters(X_scaled, test_centroids)
print(f'Labels shape: {test_labels.shape}')  # should be (200,)
print(f'Unique clusters assigned: {np.unique(test_labels)}')

In [ ]:
def update_centroids(X, labels, K):
    """
    Recompute each centroid as the mean of all points assigned to it.

    Args:
        X (np.ndarray): Data matrix.
        labels (np.ndarray): Current cluster assignments.
        K (int): Number of clusters.

    Returns:
        np.ndarray: Updated centroid array of shape (K, n_features).
                    If a cluster is empty, its centroid is kept unchanged.
    """
    new_centroids = np.zeros((K, X.shape[1]))
    for k in range(K):
        points_in_cluster = X[labels == k]
        if len(points_in_cluster) == 0:
            # Edge case: empty cluster — keep previous centroid unchanged
            new_centroids[k] = new_centroids[k]
        else:
            new_centroids[k] = points_in_cluster.mean(axis=0)
    return new_centroids



updated = update_centroids(X_scaled, test_labels, K=5)
print(f'Updated centroids shape: {updated.shape}')  # (5, 2)
print('Updated centroids (scaled):')
print(updated)

In [ ]:
def compute_distortion(X, labels, centroids):
    """
    Compute total distortion J — sum of squared Euclidean distances
    from each point to its assigned centroid.

    Args:
        X (np.ndarray): Data matrix.
        labels (np.ndarray): Cluster assignments.
        centroids (np.ndarray): Current centroid array.

    Returns:
        float: Total distortion value J.
    """
    total = 0.0
    for k in range(len(centroids)):
        points = X[labels == k]
        if len(points) > 0:
            total += np.sum((points - centroids[k]) ** 2)
    return total



distortion = compute_distortion(X_scaled, test_labels, test_centroids)
print(f'Test distortion value: {distortion:.4f}')

In [ ]:
def run_kmeans(X, K, max_iters=300, tol=1e-4, seed=42):
    """
    Orchestrate the full K-Means loop:
    Initialize centroids, then repeatedly assign and update until
    centroids move less than tol or max_iters is reached.

    Args:
        X (np.ndarray): Data matrix.
        K (int): Number of clusters.
        max_iters (int): Maximum number of iterations.
        tol (float): Convergence threshold for centroid movement.
        seed (int): Random seed for reproducibility.

    Returns:
        labels (np.ndarray): Final cluster assignments.
        centroids (np.ndarray): Final centroid positions.
        distortions (list): Distortion value recorded at each iteration.
    """
    centroids   = initialize_centroids(X, K, seed)
    distortions = []

    for i in range(max_iters):
        labels       = assign_clusters(X, centroids)
        new_cents    = update_centroids(X, labels, K)
        distortions.append(compute_distortion(X, labels, new_cents))

        # Measure how much centroids moved this iteration
        movement = np.linalg.norm(new_cents - centroids)
        centroids = new_cents

        if movement < tol:
            print(f'  Converged at iteration {i + 1}')
            break

    return labels, centroids, distortions


print('All K-Means functions defined successfully!')

In [ ]:
# Run K-Means with K=5 on standardized data
print('Running K-Means with K=5...')
labels_k5, centroids_scaled_k5, distortions_k5 = run_kmeans(X_scaled, K=5, seed=42)

print(f'\nFinal distortion (K=5): {distortions_k5[-1]:.4f}')
print(f'Number of iterations  : {len(distortions_k5)}')

# Reverse Z-score transform to get centroids in original scale
centroids_original_k5 = centroids_scaled_k5 * feature_std + feature_mean

print('\nCentroid coordinates (original scale):')
print(f'{"Cluster":<10} {"Annual Income (k$)":<22} {"Spending Score"}')
print('-' * 50)
for i, c in enumerate(centroids_original_k5):
    print(f'Cluster {i}  {c[0]:<22.2f} {c[1]:.2f}')

In [ ]:
# Run K-Means for K = 1 to 10
k_values         = list(range(1, 11))
distortion_per_k = []

print('Running K-Means for K = 1 to 10...')
print(f'{"K":<5} {"Final Distortion"}')
print('-' * 25)

for k in k_values:
    _, _, dist = run_kmeans(X_scaled, K=k, seed=42)
    distortion_per_k.append(dist[-1])
    print(f'K={k:<4} {dist[-1]:.4f}')

In [ ]:
# Elbow plot — distortion vs K with elbow point marked
plt.figure(figsize=(9, 5))
plt.plot(k_values, distortion_per_k, marker='o', color='steelblue',
         linewidth=2, markersize=7, label='Distortion')

# Mark the elbow point at K=5
plt.axvline(x=5, color='red', linestyle='--', linewidth=1.5, label='Elbow at K=5')
plt.scatter([5], [distortion_per_k[4]], color='red', s=120, zorder=5)

plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Distortion (Sum of Squared Distances)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
plt.xticks(k_values)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Clustered Scatter Plot — original (un-standardized) Annual Income vs Spending Score
# Each point colored by cluster assignment, centroids overlaid as large 'X' markers

colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']

plt.figure(figsize=(10, 6))

for k in range(5):
    cluster_points = X[labels_k5 == k]
    plt.scatter(
        cluster_points[:, 0], cluster_points[:, 1],
        color=colors[k], label=f'Cluster {k}',
        alpha=0.7, edgecolors='white', s=70
    )

# Overlay centroids in original scale as large black X markers
plt.scatter(
    centroids_original_k5[:, 0], centroids_original_k5[:, 1],
    marker='X', s=300, color='black', zorder=5, label='Centroids'
)

plt.xlabel('Annual Income (k$)', fontsize=12)
plt.ylabel('Spending Score (1-100)', fontsize=12)
plt.title('Customer Segments — K-Means Clustering (K=5)', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Convergence Plot — distortion vs iteration number for K=5
# This shows how the algorithm converged — curve should be monotonically non-increasing

plt.figure(figsize=(8, 4))
plt.plot(
    range(1, len(distortions_k5) + 1), distortions_k5,
    marker='o', color='darkorange', linewidth=2, markersize=6
)
plt.xlabel('Iteration Number', fontsize=12)
plt.ylabel('Distortion J', fontsize=12)
plt.title('Convergence Plot — K=5 (Distortion vs Iteration)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Algorithm converged in {len(distortions_k5)} iterations.')
print(f'Starting distortion : {distortions_k5[0]:.4f}')
print(f'Final distortion    : {distortions_k5[-1]:.4f}')

In [ ]:
# Print centroid values clearly so we can interpret clusters below
print('Centroid Summary (original scale):')
print(f'{"Cluster":<10} {"Annual Income (k$)":<25} {"Spending Score (1-100)"}')
print('-' * 60)
for i, c in enumerate(centroids_original_k5):
    print(f'Cluster {i}  {c[0]:<25.1f} {c[1]:.1f}')

### Cluster Interpretation

Based on the centroid values (Annual Income vs Spending Score), here is what each cluster likely represents:

- **Cluster 0 — High Income, Low Spending:** These customers earn a high annual income but spend very little at the mall. They are likely financially careful individuals

- **Cluster 1 — Low Income, High Spending:** These customers earn less but spend generously. Targeted discount deals could work well here.

- **Cluster 2 — Average Income, Average Spending:** The typical everyday mall-goer. No extreme behavior in either direction.

- **Cluster 3 — High Income, High Spending:** The most valuable customer segment. Ideal targets for premium products, exclusive memberships, and VIP offers.

- **Cluster 4 — Low Income, Low Spending:** Budget-conscious customers who visit occasionally and spend minimally.

Overall, **Cluster 3** is the most commercially valuable, while **Cluster 1** shows interesting potential despite lower income.